# 18 — Generator subset 25 % dan 50 %, update setara p100 (G0 + pemenang)
Pengganti kernel 17. Amandemen protokol (keputusan pengguna 28 Sep 2026): 20 epoch tetap memberi subset lebih sedikit update (p025 ±1/4 p100) dan checkerboard jauh lebih kuat. `amend_subset_epochs.py` menyetel p025 = 80 epoch, p050 = 40 epoch (±4.320 update, p100 = 4.340) sebelum training dan mencatatnya di `study.json`. Selebihnya sama dengan notebook 17. Tidak ada detektor; test tidak dibaca.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/texture_subsets_v2')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
# A previous stage's kernel output also carries a code copy next to study/; only the code dataset counts.
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
# Study from notebook 13 output, the filled p100 review, and the locked selection from notebook 16.
import csv, hashlib
if torch.cuda.device_count()<1: raise RuntimeError('GPU required')
print('GPUs', torch.cuda.device_count())
source=unique('study.json')
STUDY=WORK/'study'
shutil.copytree(source,STUDY)
for p in [STUDY,*STUDY.rglob('*')]: p.chmod(p.stat().st_mode|0o200)
review=list(INPUT.rglob('review_p100.csv')); selection=list(INPUT.rglob('selection.json'))
if len(review)!=1 or len(selection)!=1: raise ValueError(f'need one review_p100.csv and one selection.json; {review} {selection}')
target=STUDY/'p100/review/review.csv'
ids=lambda path:[r['blind_id'] for r in csv.DictReader(path.open(newline=''))]
if ids(review[0])!=ids(target): raise ValueError('review_p100.csv blind IDs/order differ from the study')
shutil.copyfile(review[0],target)
sel=json.loads(selection[0].read_text())
if sel['review_sha256']!=hashlib.sha256(target.read_bytes()).hexdigest(): raise ValueError('selection was made on another review')
shutil.copyfile(selection[0],STUDY/'selection.json')
WINNER=sel['winner']; print('winner',WINNER,sel['scores'])


In [ ]:
# Protocol amendment (user 2026-09-28): subset generators get the same number of updates as p100.
run(HERE/'tools/amend_subset_epochs.py','--study',STUDY,'--fractions','25','50')
print(json.dumps(json.loads((STUDY/'study.json').read_text())['amendments'][-1],indent=1))


In [ ]:
# G0 and the winner, retrained from scratch on each subset: 25% on GPU0, 50% on GPU1 in parallel.
import shlex, time
FRACTIONS={25:0,50:1}
ARMS_SUBSET=['G0',WINNER]
n_gpu=torch.cuda.device_count()
def chain(fraction,gpu):
    base=[sys.executable,str(CLI)]
    steps=[base+['train','--study',str(STUDY),'--fraction',str(fraction),'--arms',*ARMS_SUBSET,'--budget-minutes','30'],
           base+['export','--study',str(STUDY),'--dataset',str(DATASET),'--fraction',str(fraction),'--arms',*ARMS_SUBSET,'--device','cuda'],
           base+['diagnose','--study',str(STUDY),'--fraction',str(fraction),'--arms',*ARMS_SUBSET,'--device','cuda'],
           base+['review','--study',str(STUDY),'--fraction',str(fraction),'--arms',*ARMS_SUBSET]]
    env=(f"CUDA_VISIBLE_DEVICES={gpu if n_gpu>=2 else 0} PYTHONPATH={shlex.quote(str(HERE))} "
         f"MELON_DATASET={shlex.quote(str(DATASET))} MELON_ASSETS={shlex.quote(str(ASSETS))} OMP_NUM_THREADS=2")
    log=open(WORK/f'subset_p{fraction:03d}.log','w')
    return subprocess.Popen(' && '.join(env+' '+shlex.join(s) for s in steps),shell=True,cwd=str(HERE),stdout=log,stderr=subprocess.STDOUT)
started=time.time()
if n_gpu>=2: codes=[p.wait() for p in [chain(f,g) for f,g in FRACTIONS.items()]]
else: codes=[chain(f,0).wait() for f in FRACTIONS]
print('return codes',codes,'wall minutes',round((time.time()-started)/60,1))
for f in FRACTIONS: print(f'--- p{f:03d}'); print((WORK/f'subset_p{f:03d}.log').read_text()[-2500:])
if any(codes): raise SystemExit('subset stage failed')
for f in FRACTIONS:
    for arm in ARMS_SUBSET:
        d=json.loads((STUDY/f'p{f:03d}/diagnostics/{arm}.json').read_text())['synthetic']
        print(f, arm, {k:round(d[k],4) for k in ('nyquist_luma','chroma_energy','luma_energy','lbp_chi2','red_flag_count')})


In [ ]:
# Blind-review packs (<=45 MB zips + offline review_tool.html) for download; private keys never packed.
import zipfile, hashlib
PACK=WORK/'review_pack'; PACK.mkdir()
for f in FRACTIONS:
    review=STUDY/f'p{f:03d}/review'
    subprocess.run([sys.executable,str(HERE/'tools/make_texture_review_tool.py'),'--review',str(review)],check=True)
    files=sorted(p for p in review.rglob('*') if p.is_file())
    if any('PRIVATE' in p.name for p in files): raise ValueError('private key inside review folder')
    parts,current,size=[],[],0
    for p in files:
        s=p.stat().st_size
        if current and size+s>45*2**20: parts.append(current); current,size=[],0
        current.append(p); size+=s
    parts.append(current)
    manifest=dict(fraction=f,n_files=len(files),files={},parts={})
    for i,part in enumerate(parts):
        target=PACK/f'review_p{f:03d}_part{i:02d}.zip'
        with zipfile.ZipFile(target,'w',zipfile.ZIP_STORED) as z:
            for p in part:
                rel=f'p{f:03d}/review/'+str(p.relative_to(review)); z.write(p,rel)
                manifest['files'][rel]=hashlib.sha256(p.read_bytes()).hexdigest()
        manifest['parts'][target.name]=dict(n=len(part),bytes=target.stat().st_size,sha256=hashlib.sha256(target.read_bytes()).hexdigest())
    (PACK/f'review_p{f:03d}_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    print(f, len(files), 'files', {k:round(v['bytes']/2**20,1) for k,v in manifest['parts'].items()})
for f in FRACTIONS:
    for name in ('diagnostics','logs'): shutil.copytree(STUDY/f'p{f:03d}'/name, PACK/f'p{f:03d}_{name}')


In [ ]:
# Keep what the detector stage needs; drop what makes the output slow to list or download.
for sub in ('checkpoints','candidates','panels','crops'): shutil.rmtree(STUDY/'p100'/sub, ignore_errors=True)
for f in FRACTIONS:
    for ck in (STUDY/f'p{f:03d}/checkpoints').glob('*/*.pth'):
        if not ck.name.startswith('latest_'): ck.unlink()
shutil.rmtree(WORK/'code',ignore_errors=True)
total=sum(p.stat().st_size for p in WORK.rglob('*') if p.is_file())
print('output files', sum(1 for p in WORK.rglob('*') if p.is_file()), 'size MB', round(total/2**20))
